# B2-022 — Practice p19

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** variational-autoencoder

## Task

Train a tiny VAE on the seeded `mixture8d` data and certify negative-ELBO decrease, KL bookkeeping, and held-out evaluation.

**Data.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `latent_data.train_tensor("mixture8d")` of shape `(48,8)` and `latent_data.heldout_tensor("mixture8d")` of shape `(12,8)`, built from the immutable `latent_data.TRAIN_IDS["mixture8d"]` and `latent_data.HELDOUT_IDS["mixture8d"]` in stored order. Pass training rows to the model exactly as returned (same float32 values; no normalization, noise, shuffling, or concatenation), so every row the model's `forward` receives during training hashes, with `latent_data.row_sha256`, into the training part of `latent_data.ROW_SHA256["mixture8d"]`. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice.

**Loss.** Write `negative_elbo_terms(x, x_hat, mu, logvar)` and `negative_elbo(x, x_hat, mu, logvar)` with exactly the p12 contract (recon summed over features with the $\tfrac d2\log2\pi$ constant dropped, KL to $N(0,I)$ summed over latents, mean over the batch). On p12's literals, restated in the supplied cell as `P12_*`, `negative_elbo` must return `1.5288552` (`atol = rtol = 1e-6`).

**Model.** `class TinyVAE(nn.Module)` with constructor `TinyVAE(in_dim=8, hidden=16, latent=2)` creating, in this order: `self.hidden = nn.Linear(in_dim, hidden)`, `self.mu_head = nn.Linear(hidden, latent)`, `self.logvar_head = nn.Linear(hidden, latent)`, `self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))`. Methods: `encode(x)` returns `(mu, logvar)` from `h = F.relu(self.hidden(x))`; `decode(z)` returns `self.decoder(z)`; `forward(x, eps)` returns `(x_hat, mu, logvar)` with `z = mu + torch.exp(0.5 * logvar) * eps`. `forward` draws no random numbers. Call `torch.manual_seed(SEED)` immediately before `model = TinyVAE()`.

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_vae(model, batch, optimizer)` that creates `eps_generator = torch.Generator().manual_seed(SEED)` once, then performs **exactly 600** updates; each update first draws `eps = torch.randn(batch.shape[0], 2, generator=eps_generator)`, then runs `optimizer.zero_grad(set_to_none=True)`, forward, loss, `loss.backward()`, `optimizer.step()` with `model(batch, eps)` as the forward and `negative_elbo` as the loss. Full batch, no shuffling, no scheduler, no clipping, no $\beta$ other than $1$. It returns a list of 600 dicts `{"total": float, "recon": float, "kl": float}` holding the batch means from that update's forward pass. Call it as `trace = train_vae(model, train_rows(), optimizer)`.

**Held-out evaluation.** `eval_eps = torch.randn(12, 2, generator=torch.Generator().manual_seed(SEED + 1))`. Under `torch.no_grad()`, compute the held-out total, recon, and KL (batch means) with `model(heldout_rows(), eval_eps)` before training and after training.

**Certify.** `len(trace) == 600`; for every record, `abs(total - (recon + kl)) <= 1e-5` and `kl >= 0`; the mean of the last 20 totals is `<= 0.6 *` the first total; held-out total after `<= 0.6 *` held-out total before; held-out mean KL after `>= 0.1` (the latent is in use); at least one parameter changed; split hashes as in p17.
Then, in four sentences, interpret the final held-out recon and KL values, and explain why the trace is compared through a 20-update window.

**Required answer-check assertions.** Every certificate item, the p12 literal probe, and the shapes `(12,8)`, `(12,2)`, `(12,2)` of the held-out forward outputs.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

P12_X = torch.tensor([[1.0, 0.0, -1.0, 2.0], [0.5, 0.5, 0.5, 0.5], [-2.0, 1.0, 0.0, 1.0]])
P12_X_HAT = torch.tensor([[0.8, 0.1, -1.0, 1.5], [0.5, 0.0, 1.0, 0.5], [-1.0, 1.0, 0.5, 0.0]])
P12_MU = torch.tensor([[0.0, 0.0], [0.5, -0.5], [1.0, 2.0]])
P12_LOGVAR = torch.tensor([[0.0, 0.0], [-1.0, 0.0], [0.5, -0.5]])

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.